In [ ]:
#0) Install & Upload (Colab)

!pip -q install numpy pandas openpyxl

from google.colab import files
uploaded = files.upload()  # upload: data.xlsx dan W8.xlsx


Saving W8.xlsx to W8.xlsx
Saving data.xlsx to data.xlsx


In [ ]:
#1) Load data + utilitas

import numpy as np
import pandas as pd

DATA_PATH = "data.xlsx"
W8_PATH   = "W8.xlsx"

df = pd.read_excel(DATA_PATH)

# Pastikan kolom sesuai
assert {"Waktu","Lokasi","Y","Y_(t-1)"}.issubset(df.columns), df.columns

# Buat ID lokasi (0..34) berdasarkan urutan kemunculan di file
loc_order = df["Lokasi"].drop_duplicates().tolist()
loc_to_id = {name:i for i,name in enumerate(loc_order)}
df["loc_id"] = df["Lokasi"].map(loc_to_id)

# Tahun unik (data Anda urut 2022→2018; tidak masalah)
years_unique = sorted(df["Waktu"].unique())
T = len(years_unique)
n = df["loc_id"].nunique()

print("n (locations):", n)
print("T (years):", T, years_unique)
print("N rows:", len(df))
print(df.head())

n (locations): 35
T (years): 5 [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
N rows: 175
   Waktu                 Lokasi     Y  Y_(t-1)  loc_id
0   2022       District Cilacap  9.62     9.97       0
1   2022      District Banyumas  6.05     6.05       1
2   2022   District Purbalingga  5.23     6.05       2
3   2022  District Banjarnegara  6.38     5.86       3
4   2022       District Kebumen  5.92     6.03       4


In [ ]:
#2) Wy building function for W_ST (175×175) and W_bd (35×35)

def compute_Wy_ST(df, W_ST):
    """
    W_ST: 175x175, dikalikan langsung dengan vektor y (panjang 175)
    sesuai urutan baris df.
    """
    y_vec = df["Y"].astype(float).to_numpy().reshape(-1,1)  # 175x1
    Wy = (W_ST @ y_vec).ravel()
    return Wy

def compute_Wy_bd(df, W_bd):
    """
    W_bd: 35x35, dihitung per tahun:
    Wy_t = W_bd @ y_t, lalu distack sesuai urutan df.
    Asumsi: dalam setiap tahun, urutan 35 lokasi di df sama dengan urutan W_bd.
    """
    Wy = np.zeros(len(df), dtype=float)
    for yr in df["Waktu"].unique():
        idx = df.index[df["Waktu"] == yr].to_numpy()
        y_t = df.loc[idx, "Y"].astype(float).to_numpy().reshape(-1,1)  # 35x1
        if y_t.shape[0] != W_bd.shape[0]:
            raise ValueError(f"Year {yr}: expected {W_bd.shape[0]} rows, got {y_t.shape[0]}")
        Wy[idx] = (W_bd @ y_t).ravel()
    return Wy


In [ ]:
#3) Eq.(14) estimation with location fixed effect (within estimator)

def fit_eq13_fixed_effect(df_in, Wy):
    """
    Fit: y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    using within transformation (demean by location).

    Returns: rho_hat, beta_hat, fitted, residual, R2, RMSE, n_obs
    """
    dfm = df_in.copy()
    dfm["Wy"] = Wy

    # gunakan hanya baris dengan lag tersedia (2019–2022)
    dfm = dfm.dropna(subset=["Y_(t-1)"]).copy()

    y = dfm["Y"].astype(float).to_numpy()
    x1 = dfm["Wy"].astype(float).to_numpy()
    x2 = dfm["Y_(t-1)"].astype(float).to_numpy()
    g  = dfm["loc_id"].to_numpy()

    # within (demean by group)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    x1_dm = x1 - pd.Series(x1).groupby(g).transform("mean").to_numpy()
    x2_dm = x2 - pd.Series(x2).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([x1_dm, x2_dm])  # tanpa intercept (FE sudah dihapus via demeaning)

    # OLS
    XtX = X.T @ X
    Xty = X.T @ y_dm
    coef = np.linalg.solve(XtX, Xty)

    rho_hat, beta_hat = coef[0], coef[1]

    # fitted untuk versi demeaned
    yhat_dm = X @ coef
    resid_dm = y_dm - yhat_dm

    # RMSE dan R2 (within)
    rmse = float(np.sqrt(np.mean(resid_dm**2)))
    ss_res = float(np.sum(resid_dm**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(rho_hat),
        "beta_hat": float(beta_hat),
        "rmse_within": rmse,
        "r2_within": r2,
        "n_obs": int(len(dfm))
    }


In [ ]:
#4) Run for all weights on W8.xlsx

xls = pd.ExcelFile(W8_PATH)
print("Sheets:", xls.sheet_names)

W_ST_sheets = ["W_ST^Q","W_ST^Q10","W_ST^Qq","W_ST^Q10q","W_ST^Qr","W_ST^Q10r","W_bdst"]
W_bd_sheet  = "W_bd"

results = []

# 7 matriks W_ST (175x175)
for sh in W_ST_sheets:
    W_ST = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
    assert W_ST.shape == (len(df), len(df)), (sh, W_ST.shape)

    Wy = compute_Wy_ST(df, W_ST)
    out = fit_eq13_fixed_effect(df, Wy)
    out["Weight"] = sh
    out["Wy_type"] = "W_ST (175x175)"
    results.append(out)

# 1 matriks W_bd (35x35, per tahun)
W_bd = pd.read_excel(W8_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)
assert W_bd.shape == (n, n), ("W_bd", W_bd.shape)

Wy_bd = compute_Wy_bd(df, W_bd)
out = fit_eq13_fixed_effect(df, Wy_bd)
out["Weight"] = "W_bd"
out["Wy_type"] = "W_bd per-year (35x35)"
results.append(out)

res_df = pd.DataFrame(results).sort_values("Weight").reset_index(drop=True)
res_df

Sheets: ['W_ST^Q', 'W_ST^Q10', 'W_ST^Qq', 'W_ST^Q10q', 'W_ST^Qr', 'W_ST^Q10r', 'W_bdst', 'W_bd']


,rho_hat,beta_hat,rmse_within,r2_within,n_obs,Weight,Wy_type
0,0.577885,-0.550419,0.321801,0.898333,140,W_ST^Q,W_ST (175x175)
1,0.500824,0.033125,0.407228,0.837190,140,W_ST^Q10,W_ST (175x175)
2,0.568013,-0.561185,0.352132,0.878265,140,W_ST^Q10q,W_ST (175x175)
3,0.583959,-0.569461,0.350496,0.879393,140,W_ST^Q10r,W_ST (175x175)
4,0.572988,-0.563499,0.334730,0.890000,140,W_ST^Qq,W_ST (175x175)
5,0.564357,-0.540890,0.347529,0.881427,140,W_ST^Qr,W_ST (175x175)
6,0.881017,0.092796,0.703399,0.514255,140,W_bd,W_bd per-year (35x35)
7,0.567263,-1.116435,0.337379,0.888252,140,W_bdst,W_ST (175x175)


In [ ]:
#5) Monte Carlo (Block Bootstrap by Location)
### produced 95% CIs for ρ,β as well as RMSE/R2 (within) with resampling per location (block), which is most reasonable for small panels.

def block_bootstrap_eq13(df_in, Wy_full, B=499, seed=2026):
    rng = np.random.default_rng(seed)
    loc_ids = np.sort(df_in["loc_id"].unique())
    nloc = len(loc_ids)

    stats = []

    for b in range(B):
        # sample lokasi dengan replacement
        samp_locs = rng.choice(loc_ids, size=nloc, replace=True)

        # ambil baris sesuai samp_locs (gabungkan)
        parts = []
        Wy_parts = []
        for loc in samp_locs:
            sub = df_in[df_in["loc_id"] == loc].copy()
            parts.append(sub)
            Wy_parts.append(Wy_full[sub.index.to_numpy()])

        df_b = pd.concat(parts, axis=0)
        Wy_b = np.concatenate(Wy_parts, axis=0)

        try:
            out = fit_eq13_fixed_effect(df_b.reset_index(drop=True), Wy_b)
            stats.append(out)
        except np.linalg.LinAlgError:
            continue

    boot = pd.DataFrame(stats)
    def ci(x):
        return float(np.percentile(x,2.5)), float(np.percentile(x,97.5))

    summary = {
        "rho_mean": float(boot["rho_hat"].mean()),
        "rho_ci95": ci(boot["rho_hat"]),
        "beta_mean": float(boot["beta_hat"].mean()),
        "beta_ci95": ci(boot["beta_hat"]),
        "rmse_mean": float(boot["rmse_within"].mean()),
        "rmse_ci95": ci(boot["rmse_within"]),
        "r2_mean": float(boot["r2_within"].mean()),
        "r2_ci95": ci(boot["r2_within"]),
        "B_effective": int(len(boot))
    }
    return summary

# contoh untuk satu bobot:
sh = "W_ST^Q"
W_ST = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
y = compute_Wy_ST(df, W_ST)
summary = block_bootstrap_eq13(df, Wy, B=499, seed=2026)
summary

{'rho_mean': 0.566106047352173,
 'rho_ci95': (0.510879382702987, 0.6104146201814459),
 'beta_mean': -1.1142167182904523,
 'beta_ci95': (-1.2334443720584314, -0.9729658674613036),
 'rmse_mean': 0.3335013155647758,
 'rmse_ci95': (0.2876671994583527, 0.3800052865123769),
 'r2_mean': 0.8897109212545592,
 'r2_ci95': (0.8554720649050366, 0.9210507492523341),
 'B_effective': 499}

In [ ]:
bootstrap_results = []

# Bootstrap for 7 W_ST matrices (175x175)
for sh in W_ST_sheets:
    W_ST = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
    Wy_current = compute_Wy_ST(df, W_ST)
    summary = block_bootstrap_eq13(df, Wy_current, B=499, seed=2026)
    summary["Weight"] = sh
    summary["Wy_type"] = "W_ST (175x175)"
    bootstrap_results.append(summary)

# Bootstrap for 1 W_bd matrix (35x35, per year)
W_bd = pd.read_excel(W8_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)
Wy_bd_current = compute_Wy_bd(df, W_bd)
summary = block_bootstrap_eq13(df, Wy_bd_current, B=499, seed=2026)
summary["Weight"] = "W_bd"
summary["Wy_type"] = "W_bd per-year (35x35)"
bootstrap_results.append(summary)

bootstrap_df = pd.DataFrame(bootstrap_results).sort_values("Weight").reset_index(drop=True)
display(bootstrap_df)

,rho_mean,rho_ci95,beta_mean,beta_ci95,rmse_mean,rmse_ci95,r2_mean,r2_ci95,B_effective,Weight,Wy_type
0,0.576455,"(0.5213935805140514, 0.6210699137985535)",-0.548686,"(-0.6295101716548183, -0.45161194710560465)",0.318144,"(0.2763204598967717, 0.3573449666768475)",0.899488,"(0.8684638118566935, 0.9276742379684751)",499,W_ST^Q,W_ST (175x175)
1,0.501922,"(0.4203844889345484, 0.5712751079446119)",0.034015,"(-0.03924353353306179, 0.10805581612291622)",0.399369,"(0.31541322711175407, 0.4812074975880297)",0.840689,"(0.770377134945726, 0.9034769632008229)",499,W_ST^Q10,W_ST (175x175)
2,0.567186,"(0.5099030736584574, 0.6194501310850966)",-0.558174,"(-0.6530358265057484, -0.4525106299856125)",0.347801,"(0.287966486566681, 0.4001990317493389)",0.879880,"(0.8340033762579703, 0.9165628417964466)",499,W_ST^Q10q,W_ST (175x175)
3,0.583266,"(0.5245860792641515, 0.6345092635399925)",-0.566544,"(-0.6603076240253595, -0.45888880123711984)",0.346231,"(0.2881661768045427, 0.3983088442302754)",0.880754,"(0.830902930685223, 0.918133980950822)",499,W_ST^Q10r,W_ST (175x175)
4,0.572273,"(0.5181831962615722, 0.6226038405018199)",-0.560301,"(-0.6599034802850154, -0.45270796579393646)",0.330445,"(0.2813810314890121, 0.3747137796152492)",0.891826,"(0.8589962408380125, 0.9211921913839951)",499,W_ST^Qq,W_ST (175x175)
5,0.563983,"(0.5075260055068438, 0.6163334545342541)",-0.538149,"(-0.6436563591864514, -0.42580583373980035)",0.342877,"(0.2918825526663984, 0.3939050784535469)",0.883577,"(0.8485568955492108, 0.912679484394401)",499,W_ST^Qr,W_ST (175x175)
6,0.886640,"(0.7379367908409846, 1.066194048265337)",0.091167,"(-0.041874271359489935, 0.1970138654789443)",0.697441,"(0.5616891281256309, 0.838331759203048)",0.522406,"(0.4217230473363268, 0.6248162245427932)",499,W_bd,W_bd per-year (35x35)
7,0.566106,"(0.510879382702987, 0.6104146201814459)",-1.114217,"(-1.2334443720584314, -0.9729658674613036)",0.333501,"(0.2876671994583527, 0.3800052865123769)",0.889711,"(0.8554720649050366, 0.9210507492523341)",499,W_bdst,W_ST (175x175)
